In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
model = PPO.load("RLmodels/1env_ppo_2566_final", device="cpu")
model

In [3]:
fragment_pool = pd.read_csv("source/SMILES_list.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=5,
                          max_k=5,
                          max_steps=6,
                          eps=1e-8,
                          r_cap=100,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        #print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [4]:
df = generate_SMILES(model, eval_env, n_episodes=10)
df.to_csv("generated_OPSs/rl_candidates_2566_1env.csv", index=False)
df

,episode,smiles,selected_fragments,reward
0,0,Nc1ccc(OC2=C(c3ccc(-c4ccc(N)c([N+](=O)[O-])c4)...,"(O=C1OCC(O)=C1c1ccccc1, Nc1ccccc1S(N)(=O)=O, N...",20.197199
1,1,Nc1ccc(OC2=C(c3ccc(-c4ccc(N)c([N+](=O)[O-])c4)...,"(O=C1OCC(O)=C1c1ccccc1, Nc1ccccc1S(N)(=O)=O, N...",20.197199
2,2,Nc1ccc(-c2ccc3c(c2)C(=O)N(c2ccc(N)c([N+](=O)[O...,"(O=CC1NC(=O)c2ccccc21, Nc1ccccc1S(N)(=O)=O, Nc...",45.837838
3,3,Nc1ccc(-c2cc([N+](=O)[O-])c(N)cc2-c2c(N)c(S(N)...,"(Nc1ccccc1S(N)(=O)=O, Nc1ccccc1S(N)(=O)=O, Nc1...",6.751398
4,4,Nc1ccc(-c2cc([N+](=O)[O-])c(N)cc2-c2c(N)c(S(N)...,"(Nc1ccccc1S(N)(=O)=O, Nc1ccccc1S(N)(=O)=O, Nc1...",6.751398
5,5,Nc1ccc(C(=O)C2c3ccc(-c4ccc(N)c(S(N)(=O)=O)c4)c...,"(O=CC1NC(=O)c2ccccc21, Nc1ccccc1[N+](=O)[O-], ...",49.243041
6,6,Nc1ccc(-c2ccc3c(c2)C(=O)N(c2ccc(N)c([N+](=O)[O...,"(O=CC1NC(=O)c2ccccc21, Nc1ccccc1S(N)(=O)=O, Nc...",45.837838
7,7,Nc1ccc(-c2ccc3c(c2)C(=O)N(c2ccc(N)c([N+](=O)[O...,"(O=CC1NC(=O)c2ccccc21, Nc1ccccc1S(N)(=O)=O, Nc...",30.954836
8,8,Nc1ccc(OC2=C(c3ccc(-c4ccc(N)c([N+](=O)[O-])c4)...,"(O=C1OCC(O)=C1c1ccccc1, Nc1ccccc1S(N)(=O)=O, N...",20.153152
9,9,Nc1ccc(-c2ccc3c(c2)C(=O)N(c2ccc(N)c([N+](=O)[O...,"(O=CC1NC(=O)c2ccccc21, Nc1ccccc1S(N)(=O)=O, Nc...",45.837838
